In [1]:
import pandas as pd

In [2]:
import sys

In [3]:
sys.path.append('..')

In [4]:
df_questions = pd.read_csv('questions.csv')
df_questions.head()

,question,group,type
0,How do I set up LLM as a judge?,LLM evaluation,direct
1,I want to evaluate my chatbot responses,LLM evaluation,vague
2,How do I customize evaluation criteria for my ...,LLM evaluation,specific
3,What metrics are available?,Metrics and descriptors,broad
4,How do I measure text length?,Metrics and descriptors,specific descriptor


In [5]:
row = df_questions.iloc[0]
row.question

'How do I set up LLM as a judge?'

In [14]:

from tools import create_documentation_tools_cached
from doc_agent import create_agent, DocumentationAgentConfig

tools = create_documentation_tools_cached()
agent_config = DocumentationAgentConfig()

agent = create_agent(agent_config, tools)

Loading documentation tools from .cache/documentation_tools.pkl...


In [15]:
from doc_agent import AgentStreamRunner
from jaxn import JSONParserHandler

async def run_agent(agent, user_prompt, message_history=None):
    runner = AgentStreamRunner(agent, JSONParserHandler())
    result = await runner.run(user_prompt, message_history)
    return result

In [16]:
result = await run_agent(agent, row.question)

USER PROMPT (search): How do I set up LLM as a judge?
TOOL CALL (search): search({'query': 'set up LLM as a judge'})
TOOL CALL (search): get_file({'filename': 'examples/LLM_judge.mdx'})


In [17]:
from tests.cost_tracker import calculate_cost


usage = result.usage()

cost = calculate_cost(
    model_name=agent_config.model,
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens
)

In [18]:
import dataclasses
from tests.utils import collect_tools

tools = collect_tools(result.all_messages())
tools_dicts = [dataclasses.asdict(t) for t in tools]

In [19]:
rag_output = result.output

In [12]:
output = {
    'input': row.to_dict(),
    'rag_response': rag_output.model_dump(),
    'tools': tools_dicts,
    'cost': cost
}

In [20]:
async def run_eval_round(row):
    result = await run_agent(agent, row.question)

    usage = result.usage()

    cost = calculate_cost(
        model_name=agent_config.model,
        input_tokens=usage.input_tokens,
        output_tokens=usage.output_tokens
    )

    tools = collect_tools(result.all_messages())
    tools_dicts = [dataclasses.asdict(t) for t in tools]
    
    rag_output = result.output

    output = {
        'input': row.to_dict(),
        'rag_response': rag_output.model_dump(),
        'tools': tools_dicts,
        'cost': cost
    }

    return output

In [21]:
output = await run_eval_round(row)

USER PROMPT (search): How do I set up LLM as a judge?
TOOL CALL (search): search({'query': 'LLM as a judge'})
TOOL CALL (search): get_file({'filename': 'examples/LLM_judge.mdx'})
TOOL CALL (search): get_file({'filename': 'quickstart_llm.mdx'})


In [22]:
output

{'input': {'question': 'How do I set up LLM as a judge?',
  'group': 'LLM evaluation',
  'type': 'direct'},
 'rag_response': {'answer': 'To set up LLM as a judge, follow these steps:\n\n1.  **Install Evidently and Import Modules**: First, install the Evidently Python library and import the necessary modules:\n\n    ```python\n    pip install evidently\n\n    import pandas as pd\n    import os\n\n    from evidently import Dataset\n    from evidently import DataDefinition\n    from evidently import Report\n    from evidently.descriptors import LLMEval\n    from evidently.llm.templates import BinaryClassificationPromptTemplate\n    from evidently.presets import TextEvals\n    ```\n\n2.  **Set OpenAI API Key**: Provide your OpenAI API key as an environment variable:\n\n    ```python\n    os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n    ```\n\n3.  **Create the Dataset**: Prepare a Pandas DataFrame that includes the text data you want the LLM to evaluate. For instance, if you are evaluating re

In [23]:
from tqdm.auto import tqdm

In [24]:
import traceback

outputs = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    try:
        output = await run_eval_round(row)
        outputs.append(output)

    except Exception as e:
        print(f"Cannot process row {row}: {e}")
        traceback.print_exc()

  0%|          | 0/27 [00:00<?, ?it/s]

USER PROMPT (search): How do I set up LLM as a judge?
TOOL CALL (search): search({'query': 'LLM as a judge'})
TOOL CALL (search): get_file({'filename': 'examples/LLM_judge.mdx'})
TOOL CALL (search): get_file({'filename': 'quickstart_llm.mdx'})
USER PROMPT (search): I want to evaluate my chatbot responses
TOOL CALL (search): search({'query': 'evaluate chatbot responses'})
TOOL CALL (search): get_file({'filename': 'quickstart_llm.mdx'})
TOOL CALL (search): get_file({'filename': 'examples/LLM_judge.mdx'})
TOOL CALL (search): get_file({'filename': 'examples/LLM_rag_evals.mdx'})
USER PROMPT (search): How do I customize evaluation criteria for my use case?
TOOL CALL (search): search({'query': 'customize evaluation criteria'})
TOOL CALL (search): get_file({'filename': 'metrics/customize_llm_judge.mdx'})
TOOL CALL (search): search({'query': 'custom criteria evaluation templates'})
USER PROMPT (search): What metrics are available?
TOOL CALL (search): search({'query': 'metrics available'})
TOOL 

In [25]:
len(outputs)

27

In [26]:
import json

In [27]:
with open('evals_run_2026_08_05.json', 'wt') as f_out:
    json.dump(outputs, f_out, indent=2)

In [28]:
total_cost = sum([o['cost'] for o in outputs])
print(f'total cost: {total_cost:0.3f}')

total cost: 0.172
